In [ ]:
#importing libraries
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt  
import seaborn as sns
import json

In [ ]:
#importing train,test,split,logisticregression and standardScaler from sklearn
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_curve, roc_auc_score
)

In [ ]:
#loading the dataset
df = pd.read_csv('../../cleaned/titanic_cleaned.csv')

In [ ]:
df.head()

In [ ]:
#feature x and target y
X = df.drop('Survived', axis=1)
y = df['Survived']


In [ ]:
#spliting data 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
#filling NaN values
imputer = SimpleImputer(strategy='mean')

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

In [ ]:
#standardScaler normalizes features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test) 


In [ ]:
#now_creating_and_training the model
model_lr = LogisticRegression(max_iter=200, random_state=42)
model_lr.fit(X_train_scaled, y_train)

In [ ]:
#making_predictions
y_pred = model_lr.predict(X_test_scaled)
y_prob = model_lr.predict_proba(X_test_scaled)[:, 1]



In [ ]:
#counting model_accuracy 
accuracy = accuracy_score(y_test, y_pred)
auc      = roc_auc_score(y_test, y_prob)

print(f'  Accuracy : {accuracy:.4f} ({accuracy*100:.1f}%)')
print(f'  ROC-AUC  : {auc:.4f}')
print(classification_report(y_test, y_pred, target_names=['Died', 'Survived']))

In [ ]:
#confusion matrix
fig, ax = plt.subplots(figsize=(5,4))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                ax = ax,
                xticklabels=['Died', 'Survived'],
                yticklabels=['Died', 'Survived'],
                cbar=False)
ax.set_title('consfusion_matrix')    
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')


plt.tight_layout()
plt.show()              
   

In [ ]:
#roc_curve
fig, ax = plt.subplots(figsize=(5,4))

fpr, tpr, _ = roc_curve(y_test, y_prob)
ax.plot(fpr, tpr, lw=2, label=f'AUC = {auc:.2f}')
ax.plot([0, 1], [0, 1], linestyle='--')

ax.set_title('roc_curve')
ax.set_xlabel('FPR')
ax.set_ylabel('TPR')
ax.legend()

plt.show()

In [ ]:
#saving model result in json for comparison in Notebook 3
results = {
    'logistic_regression': {
        'accuracy': round(accuracy, 4),
        'roc_auc': round(auc, 4)
    }
}

with open('model_results.json', 'w') as f:
    json.dump(results, f)